## Patient Journey 04 (3.1) - Diabetes type and heart failure flags

### In short
* **3.0** answered one question per patient: *is this patient diabetic?* (`DIABETIC_FLAG`, effectively "T2D yes/no").
* **3.1** adds HF and T1D to the T2D + CKD indications, so two more questions are needed: *which diabetes type (T2D or T1D)?* and *does the patient have heart failure, and of which kind?*
* This notebook keeps the 3.0 flag unchanged and adds the new flags next to it. The 3.0 notebook `04_patient_journey_t2d_flag` is not touched, and PJ06 keeps reading its table until the 3.1 journeys go live.

### Before (3.0) and now (3.1)

| | 3.0 (`04_patient_journey_t2d_flag`) | 3.1 (this notebook) |
| --- | --- | --- |
| Diabetes flag | `DIABETIC_FLAG` = 1 if the patient ever had a T2D diagnosis **or** ever filled a diabetes drug | `T2D_FLAG_30` = the same 3.0 flag (kept for PJ06), plus `DM_TYPE` = T2D / T1D / OTHER, `T2D_FLAG`, `T1D_FLAG`, `DM_MIXED` |
| Type 1 diabetes | Not identified. A T1D patient with one T2D code, or on a drug from the 3.0 list, is counted as T2D | Own flag `T1D_FLAG` |
| Heart failure | Not available | `HF_FLAG`, `HF_TYPE`, `HF_FIRST_DATE` |
| How diagnosis codes are chosen | Word search: description contains "TYPE 2 DIABETES" | Code list from the product basket (`Kerendia_Dx_Final.csv`) plus `Kerendia_Dx_Flag_Additions.csv` for codes the basket is missing |
| Dates | All history, no end date | All history up to `date_end` (2026-09-30) |
| Output | `kerendia_nba_3_1_t2d_patient_validation`, one row per T2D patient | `kerendia_nba_3_1_dm_hf_flags_validation`, one row per patient with any flag (not written yet, `WRITE_OUTPUT = False`) |

### What the new flags mean
* **`DM_TYPE` / `T2D_FLAG` / `T1D_FLAG`**: the diabetes type of the patient. Many patients carry both T1D (E10) and T2D (E11) codes over time (miscoding is common). The type with more claim days wins; a tie goes to T2D. A patient with no T1D/T2D code but a diabetes drug from the 3.0 list is T2D, as in 3.0, except when the only drug is an SGLT2 on its own (decision D1: SGLT2s are also given for HF and CKD without diabetes).
* **`DM_MIXED`** = 1 when the patient has both T1D and T2D codes, so these patients can be reviewed or excluded later.
* **`HF_FLAG`** = 1 when the patient has a heart failure diagnosis; **`HF_FIRST_DATE`** = first HF claim, so a monthly flag is `month >= HF_FIRST_DATE`.
* **`HF_TYPE`**: claims have no ejection fraction, so the type comes from the code: systolic HF (I50.2x) = `HFREF_PROXY`, diastolic HF (I50.3x) = `HFPEF_PROXY`, combined code or both kinds over time = `MIXED`, unspecified = `UNKNOWN`.

### What changes in the numbers (all US patients)

| Flag | 3.0 | 3.1 | Why |
| --- | --- | --- | --- |
| T2D | 77,186,262 | 73,532,765 (-3,653,497) | 2,480,584 had only an SGLT2 alone and no diabetes code (D1: 2,471,874 lose the flag, 8,710 become OTHER); 1,165,299 move to T1D (more T1D than T2D claim days); 7,555 have T2D codes only after `date_end`; 59 move to OTHER |
| T1D | not available | 1,949,479 | 1,165,299 were T2D in 3.0, 784,180 had no 3.0 flag |
| Both T1D and T2D codes (`DM_MIXED`) | not available | 3,625,338 | 966,140 of them end up T1D, the rest T2D |
| HF | not available | 23,763,719 | 11,855,450 also T2D, 171,049 also T1D, 58,631 other diabetes, 11,678,589 without diabetes |

Validation patients (4,238 patients of the 10 test HCPs): T2D 2,490 (3.0) -> 2,339 (3.1): 36 are now T1D (39 T1D in total), 115 are no longer T2D (SGLT2 alone without a diabetes code, or T2D codes only after `date_end`). HF: 1,309 patients.

Patients found only through the added codes (not in the basket): T2D 61,979 (with the additions, the T2D codes are exactly the 3.0 list), T1D 19,751, HF 5,065.

**Sources**
* Dx claims: `PHCDW_DISCOVERY.STG_IQV_LAAD_DX` (until 2018-05-31, latest version of each claim) + `PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX` (from 2021-01). Same as 3.0.
* Rx claims: `PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX` with the 3.0 diabetes-drug rule (`kerendia_pcp.diabetes_product` + `diabetes_product_injection_mapping`).
* Codes: `Final Product Basket/Kerendia_Dx_Final.csv` (read only, columns `DM_TYPE`, `HF_TYPE`) + `Final Product Basket/Kerendia_Dx_Flag_Additions.csv` (116 codes the basket does not have: 38 T2D, 68 T1D, 10 HF).

**Rules**
* Claims up to `date_end` (from the 3.0 `00_config`) with a rendering provider (as 3.0).
* `DM_TYPE`: the type with more claim days (distinct service dates) wins; a tie goes to T2D (D2). No T1D/T2D code: T2D if the patient has a 3.0 diabetes drug other than an SGLT2 alone (D1), else OTHER if an other-diabetes code (E08/E09/E13).
* `DM_MIXED = 1` when the patient has both T1D and T2D codes.
* `HF_TYPE`: MIXED (combined code, or both HFrEF and HFpEF codes), HFREF_PROXY, HFPEF_PROXY, else UNKNOWN. `HF_FIRST_DATE` = first HF claim; a monthly HF flag is `month >= HF_FIRST_DATE` (built in PJ06).
* `T2D_FLAG_30` = the 3.0 `DIABETIC_FLAG` (any T2D code at any date, or a 3.0 diabetes drug), kept for PJ06.

**Steps**: 0. Rerun the 3.0 T2D flag and compare with its table. 1. Check the code lists. 2. Settings and building blocks. 3. All-US counts. 4. Validation patients with an independent recount. 5. Output (off).

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "../../00_cohort_config"

In [0]:
# 2a. Settings for the diabetes-type and HF flags (DM_HF_CONFIG). Change tables, dates or paths here only.
import os
import pandas as pd
import pyspark.sql.functions as F

_NB_DIR = "/Workspace" + os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
BASKET_DIR = os.path.normpath(os.path.join(_NB_DIR, "..", "..", "Final Product Basket"))
QUERY_OUTPUT = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
                "hcp_level_model/ops_pipeline/query_output/")

DM_HF_CONFIG = {
    # Dx claims: 3.0 sources, unchanged (history table until 2018-05, latest version of each claim; then the LAAD fact)
    "dx_history":   {"table": "PHCDW.PHCDW_DISCOVERY.STG_IQV_LAAD_DX", "until": "2018-05-31"},
    "dx_current":   {"table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_DX"},
    "diag_dim":     "PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_DIAGNOSIS",
    # Rx: 3.0 diabetes-drug rule (product table + injection mapping), unchanged
    "rx_fact":      "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX",
    "product_dim":  "PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT",
    "dm_products":  "kerendia_pcp.diabetes_product",
    "dm_injection": "kerendia_pcp.diabetes_product_injection_mapping",
    "sglt2_mono":   ["FARXIGA", "JARDIANCE", "INVOKANA", "STEGLATRO"],   # only used to report Rx-only T2D from SGLT2 alone
    # Code lists: the Dx basket (read only, the Dx journey uses it) + our additions file
    "dx_basket":    f"{BASKET_DIR}/Kerendia_Dx_Final.csv",
    "dx_additions": f"{BASKET_DIR}/Kerendia_Dx_Flag_Additions.csv",
    # Dates: new flags use claims up to date_end (00_config); the 3.0 T2D flag has no end date
    "end_date":     str(date_end)[:10],   # from 00_config
    # Tables
    "t2d_30_table": QUERY_OUTPUT + "kerendia_nba_3_1_t2d_patient_validation",   # written by 04_patient_journey_t2d_flag
    "output_path":  QUERY_OUTPUT + "kerendia_nba_3_1_dm_hf_flags_validation",
}

for _k in ["dx_basket", "dx_additions"]:
    assert os.path.exists(DM_HF_CONFIG[_k]), f"{_k} not found: {DM_HF_CONFIG[_k]}"
for _t in [DM_HF_CONFIG["dx_history"]["table"], DM_HF_CONFIG["dx_current"]["table"], DM_HF_CONFIG["diag_dim"],
           DM_HF_CONFIG["rx_fact"], DM_HF_CONFIG["product_dim"]]:
    Get_Data_Snowflakes(f"SELECT * FROM {_t} LIMIT 0").columns   # fails here if a table is renamed
print(f"Basket folder: {BASKET_DIR}\nEnd date: {DM_HF_CONFIG['end_date']}\nOutput: {DM_HF_CONFIG['output_path']}")

In [0]:
# 2b. Building blocks: Dx union (3.0 rule), the 3.0 diabetes-drug NDC list, and the code table from the basket.
def _q(values):
    return ",".join("'" + str(v).replace("'", "''") + "'" for v in values)

def dx_union_sql(cfg):
    """All Dx claim rows, exactly as the 3.0 T2D query reads them."""
    h, c = cfg["dx_history"], cfg["dx_current"]
    return f"""(SELECT DX.PATIENT_ID::VARCHAR PATIENT_ID, DX.DIAGNOSIS_CODE, DX.PROVIDER_RENDERING_ID, DX.SERVICE_DATE
      FROM {h['table']} DX
      JOIN (SELECT CLAIM_ID, MAX(LAST_DATE) LAST_DATE FROM {h['table']} GROUP BY CLAIM_ID) MX
        ON DX.CLAIM_ID = MX.CLAIM_ID AND DX.LAST_DATE = MX.LAST_DATE
      WHERE DX.SERVICE_DATE <= '{h['until']}'
      UNION ALL
      SELECT PATIENT_ID::VARCHAR, DIAGNOSIS_CODE, PROVIDER_RENDERING_ID, SERVICE_DATE FROM {c['table']})"""

def rx_t2d_ndcs(cfg):
    """3.0 rule: product in the diabetes product table, LOGIC = 1, or LOGIC = 2 and not an injection. Returns (sglt2_mono, other) NDCs."""
    prod = Get_Data_Snowflakes(f"SELECT DISTINCT NDC_CD, PRODUCT_NAME, PRODUCT_FORM FROM {cfg['product_dim']}").toPandas()
    m = (prod.merge(spark.table(cfg["dm_products"]).toPandas(), on="PRODUCT_NAME")
             .merge(spark.table(cfg["dm_injection"]).toPandas(), on="PRODUCT_FORM", how="left"))
    m = m[m.PRODUCT_FORM.notna() & (m.PRODUCT_FORM.str.strip() != "") & m.LOGIC.notna() & m.INJECTION.notna()
          & ~m.LOGIC.astype(str).isin(["null", ""]) & ~m.INJECTION.astype(str).isin(["null", ""])]
    m = m[(m.LOGIC.astype(int) == 1) | (m.INJECTION == "No")]
    sg = sorted(m[m.PRODUCT_NAME.isin(cfg["sglt2_mono"])].NDC_CD.unique())
    return sg, sorted(set(m.NDC_CD) - set(sg)), m

def read_flag_codes(cfg):
    """One row per code: GRP = T2D / T1D / OTHER (diabetes type) or HF; HF_TYPE for HF codes; SRC = B (basket) or A (additions)."""
    rows = []
    for path, src in [(cfg["dx_basket"], "B"), (cfg["dx_additions"], "A")]:
        b = pd.read_csv(path, dtype=str).fillna("")
        assert set(b.MATCH_TYPE) <= {"CODE", "LIST"}, f"{path}: MATCH_TYPE must be CODE or LIST"
        b = b.assign(CODE=b.MATCH_VALUE.str.split("|")).explode("CODE")
        b["CODE"] = b.CODE.str.strip()
        for _, r in b.iterrows():
            if r.DM_TYPE in ("T1D", "T2D", "OTHER"):
                rows.append((r.CODE, r.DM_TYPE, "", src, r.DIAGNOSIS_DESCRIPTION))
            elif r.HF_TYPE:
                rows.append((r.CODE, "HF", r.HF_TYPE, src, r.DIAGNOSIS_DESCRIPTION))
    codes = pd.DataFrame(rows, columns=["CODE", "GRP", "HF_TYPE", "SRC", "DESCRIPTION"])
    dup = codes[codes.CODE.duplicated(keep=False)]
    assert dup.empty, f"Code in the basket and in the additions (or twice): {sorted(dup.CODE.unique())}"
    assert set(codes.HF_TYPE) <= {"", "HFPEF_PROXY", "HFREF_PROXY", "COMBINED", "UNKNOWN"}, "Unknown HF_TYPE"
    bad = codes[(codes.GRP == "HF") & codes.DESCRIPTION.str.upper().str.contains("WITHOUT")]
    assert bad.empty, f"HF code whose description says WITHOUT: {bad.CODE.tolist()}"
    return codes

In [0]:
# 3a. The flag query: one row per patient, built from DM_HF_CONFIG and the code table. Runs in Snowflake.
def build_flags_query(cfg, codes, ndc_sglt2, ndc_other, patients=None, summary=False):
    """
    Per patient (claims up to end_date, rendering provider present, as 3.0):
      T2D_DX_DAYS / T1D_DX_DAYS / OTHER_DM_DX_DAYS = distinct service dates with a code of that type
      DM_TYPE  = T1D if T1D days > T2D days; T2D if T2D days >= T1D days (> 0);
                 no T1D/T2D code: T2D if a 3.0 diabetes drug other than an SGLT2 alone (decision D1),
                 else OTHER if an other-diabetes code
      DM_MIXED = 1 when both T1D and T2D codes are present
      HF_TYPE  = MIXED (combined code, or both HFrEF and HFpEF codes), HFREF_PROXY, HFPEF_PROXY, else UNKNOWN
      T2D_FLAG_30 = the 3.0 flag (any T2D code at any date, or a 3.0 diabetes drug)
    summary=True returns counts by flag combination instead of patient rows.
    """
    end = cfg["end_date"]
    vals = ",".join(f"({_q([r.CODE, r.GRP, r.HF_TYPE, r.SRC])})" for r in codes.itertuples())
    pat_dx = f"AND DX.PATIENT_ID IN ({_q(patients)})" if patients is not None else ""
    pat_rx = f"AND PATIENT_ID::VARCHAR IN ({_q(patients)})" if patients is not None else ""
    d = lambda cond: f"COUNT(DISTINCT IFF({cond} AND SERVICE_DATE <= '{end}', SERVICE_DATE, NULL))"
    def mx(cond, dated=True):
        date_cond = f" AND SERVICE_DATE <= '{end}'" if dated else ""
        return f"MAX(IFF({cond}{date_cond}, 1, 0))"
    sql = f"""
WITH codes AS (SELECT column1 CODE, column2 GRP, column3 HF_TYPE, column4 SRC FROM VALUES {vals}),
dx AS (SELECT DX.PATIENT_ID, DX.SERVICE_DATE, C.GRP, C.HF_TYPE, C.SRC
       FROM {dx_union_sql(cfg)} DX JOIN codes C ON DX.DIAGNOSIS_CODE = C.CODE
       WHERE DX.PROVIDER_RENDERING_ID IS NOT NULL AND DX.PATIENT_ID IS NOT NULL {pat_dx}),
p AS (SELECT PATIENT_ID,
        {d("GRP = 'T2D'")} T2D_DX_DAYS, {d("GRP = 'T1D'")} T1D_DX_DAYS, {d("GRP = 'OTHER'")} OTHER_DM_DX_DAYS,
        {mx("GRP = 'T2D'", False)} T2D_DX_ANY_DATE, {mx("GRP = 'T2D' AND SRC = 'B'", False)} T2D_DX_BASKET,
        {mx("GRP = 'T1D' AND SRC = 'B'")} T1D_DX_BASKET, {mx("GRP = 'HF' AND SRC = 'B'")} HF_DX_BASKET,
        {mx("GRP = 'HF'")} HF_FLAG,
        MIN(IFF(GRP = 'HF' AND SERVICE_DATE <= '{end}', SERVICE_DATE, NULL)) HF_FIRST_DATE,
        {mx("HF_TYPE = 'HFREF_PROXY'")} HFREF, {mx("HF_TYPE = 'HFPEF_PROXY'")} HFPEF, {mx("HF_TYPE = 'COMBINED'")} HFCOMB
      FROM dx GROUP BY 1),
rx AS (SELECT PATIENT_ID::VARCHAR PATIENT_ID, MAX(IFF(NDC_CD IN ({_q(ndc_other)}), 1, 0)) RX_T2D_NON_SGLT2
       FROM {cfg['rx_fact']} WHERE PATIENT_ID IS NOT NULL AND NDC_CD IN ({_q(ndc_sglt2 + ndc_other)}) {pat_rx} GROUP BY 1),
a AS (SELECT COALESCE(p.PATIENT_ID, rx.PATIENT_ID) PATIENT_ID, p.* EXCLUDE PATIENT_ID,
             IFF(rx.PATIENT_ID IS NULL, 0, 1) RX_T2D, COALESCE(rx.RX_T2D_NON_SGLT2, 0) RX_T2D_NON_SGLT2
      FROM p FULL OUTER JOIN rx ON p.PATIENT_ID = rx.PATIENT_ID),
f AS (SELECT PATIENT_ID,
        IFF(COALESCE(T2D_DX_ANY_DATE, 0) = 1 OR RX_T2D = 1, 1, 0) T2D_FLAG_30,
        CASE WHEN COALESCE(T1D_DX_DAYS, 0) = 0 AND COALESCE(T2D_DX_DAYS, 0) = 0
               THEN CASE WHEN RX_T2D_NON_SGLT2 = 1 THEN 'T2D' WHEN COALESCE(OTHER_DM_DX_DAYS, 0) > 0 THEN 'OTHER' END
             WHEN T1D_DX_DAYS > T2D_DX_DAYS THEN 'T1D' ELSE 'T2D' END DM_TYPE,
        IFF(COALESCE(T1D_DX_DAYS, 0) > 0 AND COALESCE(T2D_DX_DAYS, 0) > 0, 1, 0) DM_MIXED,
        COALESCE(T2D_DX_DAYS, 0) T2D_DX_DAYS, COALESCE(T1D_DX_DAYS, 0) T1D_DX_DAYS, RX_T2D, RX_T2D_NON_SGLT2,
        COALESCE(HF_FLAG, 0) HF_FLAG,
        CASE WHEN COALESCE(HF_FLAG, 0) = 0 THEN NULL WHEN HFCOMB = 1 OR (HFREF = 1 AND HFPEF = 1) THEN 'MIXED'
             WHEN HFREF = 1 THEN 'HFREF_PROXY' WHEN HFPEF = 1 THEN 'HFPEF_PROXY' ELSE 'UNKNOWN' END HF_TYPE,
        HF_FIRST_DATE,
        COALESCE(T2D_DX_ANY_DATE, 0) T2D_DX_ANY_DATE, COALESCE(T2D_DX_BASKET, 0) T2D_DX_BASKET,
        COALESCE(T1D_DX_BASKET, 0) T1D_DX_BASKET, COALESCE(HF_DX_BASKET, 0) HF_DX_BASKET
      FROM a)
"""
    if summary:
        return sql + """SELECT T2D_FLAG_30, DM_TYPE, DM_MIXED, IFF(T2D_DX_DAYS > 0, 1, 0) HAS_T2D_DX, IFF(T1D_DX_DAYS > 0, 1, 0) HAS_T1D_DX,
  IFF(T1D_DX_DAYS > 0 AND T1D_DX_DAYS = T2D_DX_DAYS, 1, 0) DM_TIE, RX_T2D, RX_T2D_NON_SGLT2, HF_FLAG, HF_TYPE,
  T2D_DX_ANY_DATE, T2D_DX_BASKET, T1D_DX_BASKET, HF_DX_BASKET, COUNT(*) PATIENTS FROM f GROUP BY ALL"""
    return sql + """SELECT PATIENT_ID, T2D_FLAG_30, DM_TYPE, IFF(DM_TYPE = 'T2D', 1, 0) T2D_FLAG, IFF(DM_TYPE = 'T1D', 1, 0) T1D_FLAG,
  DM_MIXED, T2D_DX_DAYS, T1D_DX_DAYS, RX_T2D, RX_T2D_NON_SGLT2, HF_FLAG, HF_TYPE, HF_FIRST_DATE FROM f
WHERE T2D_FLAG_30 = 1 OR DM_TYPE IS NOT NULL OR HF_FLAG = 1"""

### Step 0. 3.0 T2D flag rerun
Same rule as `04_patient_journey_t2d_flag` (T2D word match on the diagnosis description + 3.0 diabetes drugs), written as one Snowflake query: the drug filter runs in Snowflake instead of pulling every (patient, NDC) pair into Spark.

**Result (development run, 2026-10-30):** 77,186,262 patients, the same patients as the written 3.0 table (0 only in the rerun, 0 only in the table). 2,024 NDCs from 70 products.

In [0]:
# 0. 3.0 T2D flag (same rule as 04_patient_journey_t2d_flag) as one Snowflake query, compared with the written 3.0 table
ndc_sglt2, ndc_other, rx_products = rx_t2d_ndcs(DM_HF_CONFIG)
q_t2d_30 = f"""
SELECT DX.PATIENT_ID FROM {dx_union_sql(DM_HF_CONFIG)} DX
JOIN (SELECT DISTINCT DIAGNOSIS_CODE, DIAGNOSIS_DESCRIPTION FROM {DM_HF_CONFIG['diag_dim']}) CD ON DX.DIAGNOSIS_CODE = CD.DIAGNOSIS_CODE
WHERE UPPER(CD.DIAGNOSIS_DESCRIPTION) LIKE '%TYPE 2 DIABETES%' AND DX.PROVIDER_RENDERING_ID IS NOT NULL
UNION
SELECT PATIENT_ID::VARCHAR FROM {DM_HF_CONFIG['rx_fact']}
WHERE PATIENT_ID IS NOT NULL AND NDC_CD IN ({_q(ndc_sglt2 + ndc_other)})"""
df_t2d_30_new = Get_Data_Snowflakes(q_t2d_30)
df_t2d_30_old = spark.read.format("delta").load(DM_HF_CONFIG["t2d_30_table"]).select(F.col("PATIENT_ID").cast("string")).distinct()
n_new, n_old = df_t2d_30_new.count(), df_t2d_30_old.count()
n_both = df_t2d_30_new.join(df_t2d_30_old, "PATIENT_ID").count()
print(f"3.0 diabetes-drug NDCs: {len(ndc_sglt2) + len(ndc_other):,} ({rx_products.PRODUCT_NAME.nunique()} products)")
print(f"T2D patients rerun: {n_new:,} | written 3.0 table: {n_old:,} | in both: {n_both:,} | only rerun: {n_new - n_both:,} | only written: {n_old - n_both:,}")

### Step 1. Code lists
The basket is compared with a word search of the diagnosis dimension (ICD-10 only; HF excludes descriptions with WITHOUT, such as I11.9 and I13.1x).

**Result (development run):**

| Flag | Word match | Basket | Added | Not covered |
| --- | --- | --- | --- | --- |
| T2D | 124 | 86 | 38 | 0 |
| T1D | 120 | 52 | 68 | 0 |
| HF | 35 | 26 | 10 | 0 (I50.1 is in the basket, not in the word match) |

* T2D: the 38 added codes are the ones the 3.0 word match uses but the basket does not (E11 parent codes, retinopathy eye-side codes such as E11.3393 with 447,260 patients, O24.1x pre-existing T2D in pregnancy). With them, T2D codes equal the 3.0 list.
* T1D: the 68 added codes are E10 parent and eye-side codes and O24.0x (pre-existing T1D in pregnancy).
* HF: 10 added codes (I50 / I50.2 / I50.3 / I50.4 / I50.8 / I50.81 parents, I50.83 high-output HF, I97.13x post-procedure HF).
* ICD-9 codes (2013-2015) are not used, as in 3.0 (the 3.0 word match does not find them because ICD-9 says TYPE II). Patients with only ICD-9 codes: T2D 4,347,621, T1D 1,140,938, HF 3,127,750.
* Dx claims have a gap from 2018-06 to 2020-12 (history table ends 2018-05-31, LAAD fact starts 2021-01-02).

In [0]:
# 1. Code lists: basket + additions against a word search of the diagnosis dimension
codes = read_flag_codes(DM_HF_CONFIG)
dim = (Get_Data_Snowflakes(f"SELECT DISTINCT DIAGNOSIS_CODE, DIAGNOSIS_DESCRIPTION FROM {DM_HF_CONFIG['diag_dim']}")
       .toPandas().drop_duplicates("DIAGNOSIS_CODE"))
desc = dim.DIAGNOSIS_DESCRIPTION.str.upper()
icd10 = dim.DIAGNOSIS_CODE.str.match(r"^[A-Z]")
word = {"T2D": set(dim[icd10 & desc.str.contains("TYPE 2 DIABETES")].DIAGNOSIS_CODE),
        "T1D": set(dim[icd10 & desc.str.contains("TYPE 1 DIABETES")].DIAGNOSIS_CODE),
        "HF":  set(dim[icd10 & desc.str.contains("HEART FAILURE") & ~desc.str.contains("WITHOUT")].DIAGNOSIS_CODE)}
rows = []
for g in ["T2D", "T1D", "HF"]:
    b = set(codes[(codes.GRP == g) & (codes.SRC == "B")].CODE)
    a = set(codes[(codes.GRP == g) & (codes.SRC == "A")].CODE)
    rows.append({"FLAG": g, "WORD_MATCH_CODES": len(word[g]), "BASKET_CODES": len(b), "ADDED_CODES": len(a),
                 "WORD_MATCH_NOT_COVERED": ", ".join(sorted(word[g] - b - a)), "BASKET_NOT_IN_WORD_MATCH": ", ".join(sorted(b - word[g]))})
display(pd.DataFrame(rows))
print("Codes by flag and source (B = Kerendia_Dx_Final.csv, A = Kerendia_Dx_Flag_Additions.csv):")
display(codes.groupby(["GRP", "SRC"]).size().rename("codes").reset_index())
assert word["T2D"] == set(codes[codes.GRP == "T2D"].CODE), "T2D codes differ from the 3.0 word match"

### Step 3. All US patients
Counts by flag, aggregated in Snowflake (about 5 minutes).

**Result (development run):**

| Measure | Patients |
| --- | --- |
| T2D flag 3.0 | 77,186,262 |
| T2D code from the basket only (without additions) | 54,253,172 (vs 54,315,151 with additions: 61,979 more) |
| DM_TYPE = T2D (new T2D_FLAG) | 73,532,765 |
| of them by Rx only (no T1D/T2D code) | 20,196,245 |
| SGLT2 alone and no T1D/T2D code: T2D in 3.0, not T2D now (D1) | 2,480,584 (1,095,883 with an HF code) |
| DM_TYPE = T1D (new T1D_FLAG) | 1,949,479 |
| of them T2D in 3.0 | 1,165,299 |
| of them only through added codes | 19,751 |
| DM_TYPE = OTHER | 342,430 |
| DM_MIXED = 1 | 3,625,338 (966,140 primary T1D; 143,765 ties set to T2D) |
| T2D in 3.0 with no DM_TYPE now (SGLT2 alone, or T2D codes only after date_end) | 2,479,429 |
| HF_FLAG = 1 | 23,763,719 (5,065 only through added codes) |

HF_TYPE: HFPEF_PROXY 6,170,578; HFREF_PROXY 4,400,846; MIXED 5,357,248; UNKNOWN 7,835,047. HF patients by DM_TYPE: T2D 11,855,450; T1D 171,049; OTHER 58,631; no diabetes 11,678,589.

In [0]:
# 3b. All US patients: counts by flag combination (aggregated in Snowflake, no patient rows pulled)
summary = Get_Data_Snowflakes(build_flags_query(DM_HF_CONFIG, codes, ndc_sglt2, ndc_other, summary=True)).toPandas()
for _c in summary.columns:
    if _c not in ("DM_TYPE", "HF_TYPE"):
        summary[_c] = summary[_c].astype(int)
S = summary.fillna({"DM_TYPE": "none", "HF_TYPE": "none"})
n = lambda mask: int(S.loc[mask, "PATIENTS"].sum())
rx_only = (S.DM_TYPE == "T2D") & (S.HAS_T2D_DX == 0) & (S.HAS_T1D_DX == 0)
sglt2_alone = (S.RX_T2D == 1) & (S.RX_T2D_NON_SGLT2 == 0) & (S.HAS_T2D_DX == 0) & (S.HAS_T1D_DX == 0)
overview = pd.DataFrame([
    ("Patients with any flag code or 3.0 diabetes drug", n(S.PATIENTS > 0)),
    ("T2D flag 3.0 (any T2D code, any date, or 3.0 diabetes drug)", n(S.T2D_FLAG_30 == 1)),
    ("  T2D code, any date (basket + additions)", n(S.T2D_DX_ANY_DATE == 1)),
    ("  T2D code from the basket only", n(S.T2D_DX_BASKET == 1)),
    ("  3.0 diabetes drug", n(S.RX_T2D == 1)),
    ("DM_TYPE = T2D (new T2D_FLAG)", n(S.DM_TYPE == "T2D")),
    ("  by Rx only (no T1D/T2D code)", n(rx_only)),
    ("SGLT2 alone, no T1D/T2D code: T2D in 3.0, not T2D now (D1)", n(sglt2_alone)),
    ("  of them with an HF code", n(sglt2_alone & (S.HF_FLAG == 1))),
    ("DM_TYPE = T1D (new T1D_FLAG)", n(S.DM_TYPE == "T1D")),
    ("  T2D in 3.0", n((S.DM_TYPE == "T1D") & (S.T2D_FLAG_30 == 1))),
    ("  only through added codes", n((S.DM_TYPE == "T1D") & (S.T1D_DX_BASKET == 0))),
    ("DM_TYPE = OTHER (only other-diabetes codes)", n(S.DM_TYPE == "OTHER")),
    ("DM_MIXED = 1 (T1D and T2D codes)", n(S.DM_MIXED == 1)),
    ("  primary type T1D", n((S.DM_MIXED == 1) & (S.DM_TYPE == "T1D"))),
    ("  tie, set to T2D", n(S.DM_TIE == 1)),
    ("T2D in 3.0, no DM_TYPE now (SGLT2 alone, or T2D codes only after end date)", n((S.T2D_FLAG_30 == 1) & (S.DM_TYPE == "none"))),
    ("HF_FLAG = 1", n(S.HF_FLAG == 1)),
    ("  only through added codes", n((S.HF_FLAG == 1) & (S.HF_DX_BASKET == 0))),
], columns=["MEASURE", "PATIENTS"])
display(overview)
display(pd.pivot_table(S[S.HF_FLAG == 1], index="HF_TYPE", columns="DM_TYPE", values="PATIENTS", aggfunc="sum", fill_value=0, margins=True, margins_name="TOTAL"))

### Step 4. Validation patients
The 4,238 patients of the lab, Dx and Rx validation tables. The Snowflake flags are checked against a pandas recount from the raw claim rows.

**Result (development run):** 0 differences for DM_TYPE, HF_TYPE and HF_FIRST_DATE.

| Measure | Patients |
| --- | --- |
| T2D flag 3.0 | 2,490 |
| T2D_FLAG (new) | 2,339 (36 now T1D, 115 no longer T2D) |
| T1D_FLAG | 39 (36 were T2D in 3.0) |
| DM_MIXED | 205 |
| HF_FLAG | 1,309 (MIXED 424, HFPEF_PROXY 420, UNKNOWN 315, HFREF_PROXY 150) |

In [0]:
# 4. Validation patients (patients of the lab, Dx and Rx validation tables): flags, checked against a pandas recount from the raw claims
val_patients = set()
for _t in ["kerendia_nba_3_1_lab_journey_validation", "kerendia_nba_3_1_dx_journey_validation", "kerendia_nba_3_1_rx_journey_validation"]:
    val_patients |= {r[0] for r in spark.read.format("delta").load(QUERY_OUTPUT + _t).select(F.col("PATIENT_ID").cast("string")).distinct().collect()}
val_patients = sorted(val_patients)
df_val = Get_Data_Snowflakes(build_flags_query(DM_HF_CONFIG, codes, ndc_sglt2, ndc_other, patients=val_patients)).toPandas()
assert not df_val.PATIENT_ID.duplicated().any(), "Duplicate patients in the flag output"

# Independent recount in pandas from the raw claim rows (same rules, different code path)
_end = pd.Timestamp(DM_HF_CONFIG["end_date"])
_raw = Get_Data_Snowflakes(f"""SELECT DX.PATIENT_ID, DX.SERVICE_DATE, DX.DIAGNOSIS_CODE FROM {dx_union_sql(DM_HF_CONFIG)} DX
    WHERE DX.PATIENT_ID IN ({_q(val_patients)}) AND DX.PROVIDER_RENDERING_ID IS NOT NULL""").toPandas()
_rx = set(Get_Data_Snowflakes(f"""SELECT DISTINCT PATIENT_ID::VARCHAR PATIENT_ID FROM {DM_HF_CONFIG['rx_fact']}
    WHERE PATIENT_ID::VARCHAR IN ({_q(val_patients)}) AND NDC_CD IN ({_q(ndc_other)})""").toPandas().PATIENT_ID)   # D1: SGLT2 alone does not count
_cm = codes.set_index("CODE")
_raw = _raw[_raw.DIAGNOSIS_CODE.isin(_cm.index)]
_raw = _raw.assign(GRP=_raw.DIAGNOSIS_CODE.map(_cm.GRP), HT=_raw.DIAGNOSIS_CODE.map(_cm.HF_TYPE), SERVICE_DATE=pd.to_datetime(_raw.SERVICE_DATE))
_raw = _raw[_raw.SERVICE_DATE <= _end]
_days = (_raw[_raw.GRP.isin(["T1D", "T2D", "OTHER"])].groupby(["PATIENT_ID", "GRP"]).SERVICE_DATE.nunique()
         .unstack(fill_value=0).reindex(columns=["T1D", "T2D", "OTHER"], fill_value=0))
def _dm(pid):
    t1, t2, o = _days.loc[pid] if pid in _days.index else (0, 0, 0)
    if t1 == 0 and t2 == 0:
        return "T2D" if pid in _rx else ("OTHER" if o > 0 else None)
    return "T1D" if t1 > t2 else "T2D"
def _hf(t):
    if "COMBINED" in t or {"HFREF_PROXY", "HFPEF_PROXY"} <= t: return "MIXED"
    return "HFREF_PROXY" if "HFREF_PROXY" in t else "HFPEF_PROXY" if "HFPEF_PROXY" in t else "UNKNOWN"
_hfp = _raw[_raw.GRP == "HF"].groupby("PATIENT_ID").agg(T=("HT", set), D=("SERVICE_DATE", "min"))
chk = pd.DataFrame({"PATIENT_ID": val_patients})
chk["DM_PD"] = chk.PATIENT_ID.map(_dm)
chk["HF_PD"] = chk.PATIENT_ID.map(_hfp["T"].map(_hf))
chk["HFD_PD"] = chk.PATIENT_ID.map(_hfp["D"])
chk = chk.merge(df_val, on="PATIENT_ID", how="left")
_d0 = pd.Timestamp(0)
print("Differences, Snowflake vs pandas recount -> DM_TYPE:", int((chk.DM_PD.fillna("-") != chk.DM_TYPE.fillna("-")).sum()),
      "| HF_TYPE:", int((chk.HF_PD.fillna("-") != chk.HF_TYPE.fillna("-")).sum()),
      "| HF_FIRST_DATE:", int((pd.to_datetime(chk.HFD_PD).fillna(_d0) != pd.to_datetime(chk.HF_FIRST_DATE).fillna(_d0)).sum()))

_v = df_val.copy()
for _c in ["T2D_FLAG_30", "T2D_FLAG", "T1D_FLAG", "DM_MIXED", "HF_FLAG"]:
    _v[_c] = _v[_c].astype(int)
display(pd.DataFrame([
    ("Validation patients", len(val_patients)),
    ("T2D flag 3.0", _v.T2D_FLAG_30.sum()),
    ("T2D_FLAG (new)", _v.T2D_FLAG.sum()),
    ("T1D_FLAG", _v.T1D_FLAG.sum()),
    ("  of them T2D in 3.0", ((_v.T1D_FLAG == 1) & (_v.T2D_FLAG_30 == 1)).sum()),
    ("DM_MIXED", _v.DM_MIXED.sum()),
    ("HF_FLAG", _v.HF_FLAG.sum()),
] + [(f"  HF_TYPE = {k}", n) for k, n in _v.HF_TYPE.value_counts().items()], columns=["MEASURE", "PATIENTS"]))

In [0]:
# 5. Output: one row per patient with any flag. Written only when WRITE_OUTPUT = True.
#    Columns: PATIENT_ID, T2D_FLAG_30 (= 3.0 DIABETIC_FLAG, for PJ06), DM_TYPE, T2D_FLAG, T1D_FLAG, DM_MIXED,
#             T2D_DX_DAYS, T1D_DX_DAYS, RX_T2D, RX_T2D_NON_SGLT2, HF_FLAG, HF_TYPE, HF_FIRST_DATE
#    End-to-end run 2026-10-30: written to the dated run folder (RUN_OUTPUT_DIR); PJ06 still reads the 3.0 T2D table.
WRITE_OUTPUT = True
RUN_OUTPUT_DIR = QUERY_OUTPUT + "validation_2026_10_30/"
DM_HF_CONFIG["output_path"] = RUN_OUTPUT_DIR + "kerendia_nba_3_1_dm_hf_flags_validation"
df_flags = Get_Data_Snowflakes(build_flags_query(DM_HF_CONFIG, codes, ndc_sglt2, ndc_other))
if WRITE_OUTPUT:
    df_flags.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(DM_HF_CONFIG["output_path"])
    print(f"Written {spark.read.format('delta').load(DM_HF_CONFIG['output_path']).count():,} rows to {DM_HF_CONFIG['output_path']}")
else:
    print("WRITE_OUTPUT = False: nothing written. PJ06 keeps reading the 3.0 T2D table.")

### Decisions (taken 2026-10-30, open for team review)
The user asked to take the recommended option on each question and document it. Details and reasons: `NBA_3_1_DM_HF_Flags_Plan.md` and `NBA_3_1_DM_HF_Flags_Client_Summary.md` in this folder.

| # | Question | Decision | Effect |
| --- | --- | --- | --- |
| D1 | Patients with no diabetes code whose only 3.0 diabetes drug is an SGLT2 alone (Farxiga, Jardiance, Invokana, Steglatro) | **Not T2D.** SGLT2s are also given for HF and CKD without diabetes. `T2D_FLAG_30` keeps them; `RX_T2D_NON_SGLT2` = 0 shows them in the output | 2,480,584 patients (1,095,883 with HF) leave T2D |
| D2 | Patients with both T1D and T2D codes | **Keep the rule:** more claim days wins, tie goes to T2D, `DM_MIXED` = 1 | 3,625,338 mixed; 966,140 T1D; 143,765 ties |
| D3 | Insulins in the 3.0 drug list (Tresiba, Fiasp, Afrezza) | **Keep as in 3.0.** They decide only when there is no T1D/T2D code | No change |
| D4 | eGFR / CKD eligibility for T1D | **Not decided here:** this belongs to PJ06 (eligibility per indication). Proposal: same CKD stage rule as T2D until clinical input | None in PJ04 |
| D5 | ICD-9 codes (2013-2015) | **Not used,** as in 3.0. Only claims older than 10 years, and the flags look at current patients | 4,347,621 T2D, 1,140,938 T1D, 3,127,750 HF patients with ICD-9 only stay out |
| D6 | Additions file | **Keep separate** until the Dx 3.1 journey is live, then merge into `Kerendia_Dx_Final.csv` with Saurabh (the Dx journey checks its basket against the original list) | None |